# Lesson 7A: Ensemble Methods Theory

<a name="introduction"></a>
## Introduction

A single decision tree is easy to interpret but prone to high variance — small
changes in the training data can produce very different trees. Ensemble
methods combine many learners to trade a small amount of bias for a large
reduction in variance (bagging), or combine many weak learners sequentially so
each corrects the previous ones' mistakes (boosting).

Both families start from the same diagnostic tool: the **bias-variance
decomposition**, which splits a model's expected prediction error into pieces
that respond differently to ensembling.

What makes ensemble methods distinctive:
1. **Bagging** (bootstrap aggregating): train many high-variance models on
   resampled data and average their predictions to cancel out variance
2. **Random Forest**: bagging plus random feature subsampling, which
   decorrelates the trees so averaging removes even more variance
3. **Boosting**: train models sequentially, each one focusing on the previous
   ensemble's mistakes, trading a small amount of bias reduction into a
   large one
4. **Gradient boosting**: generalizes boosting to any differentiable loss by
   fitting each new model to the negative gradient (residual) of the loss

In this lesson, we'll:
1. Derive the bias-variance decomposition for squared-error loss
2. Derive why bootstrap averaging reduces variance, and why Random Forest reduces it further
3. Derive AdaBoost's weight-update rule from exponential loss minimization
4. Derive the general gradient boosting framework as iterative residual fitting
5. Implement AdaBoost from scratch using decision stumps
6. Compare a single tree, bagging, Random Forest, and AdaBoost empirically

Lesson 7b applies gradient boosting via XGBoost and LightGBM with hyperparameter tuning.


## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [The Bias-Variance Decomposition](#the-bias-variance-decomposition)
   - [Derivation](#bv-derivation)
   - [Interpretation](#bv-interpretation)
4. [Bagging](#bagging)
   - [Bootstrap Sampling](#bootstrap-sampling)
   - [Why Averaging Reduces Variance](#why-averaging-reduces-variance)
5. [Random Forest: Decorrelating the Ensemble](#random-forest-decorrelating-the-ensemble)
6. [AdaBoost](#adaboost)
   - [The Boosting Idea](#the-boosting-idea)
   - [Weighted Weak Learners](#weighted-weak-learners)
   - [Deriving the Weight-Update Rule](#deriving-the-weight-update-rule)
   - [Connection to Exponential Loss](#connection-to-exponential-loss)
   - [Training Error Bound](#training-error-bound)
7. [Gradient Boosting](#gradient-boosting)
   - [Boosting as Functional Gradient Descent](#boosting-as-functional-gradient-descent)
   - [Fitting Residuals](#fitting-residuals)
8. [From-Scratch Implementation](#from-scratch-implementation)
   - [Decision Stumps](#decision-stumps)
   - [AdaBoost in NumPy](#adaboost-in-numpy)
9. [Visualization and Interpretation](#visualization-and-interpretation)
   - [Bias-Variance Tradeoff in Practice](#bias-variance-tradeoff-in-practice)
   - [AdaBoost Weight Evolution](#adaboost-weight-evolution)
10. [Comparing Ensemble Approaches](#comparing-ensemble-approaches)
11. [Conclusion](#conclusion)
    - [Key Insights](#key-insights-3)
    - [When to Use Each Approach](#when-to-use-each-approach)
    - [Further Reading](#further-reading-3)


<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


<a name="the-bias-variance-decomposition"></a>
## The Bias-Variance Decomposition

<a name="bv-derivation"></a>
### Derivation

Consider a regression problem $y = f(\mathbf{x}) + \epsilon$ where $\epsilon$ is
noise with $\mathbb{E}[\epsilon] = 0$ and $\text{Var}(\epsilon) = \sigma^2$. Let
$\hat{f}(\mathbf{x})$ be a model trained on a random training set, so
$\hat{f}(\mathbf{x})$ is itself a random variable (it varies with the training
sample). The expected squared prediction error at a point $\mathbf{x}$, averaged
over training sets, is:

$$\mathbb{E}\left[(y - \hat{f}(\mathbf{x}))^2\right] = \mathbb{E}\left[(f(\mathbf{x}) + \epsilon - \hat{f}(\mathbf{x}))^2\right]$$

Adding and subtracting $\mathbb{E}[\hat{f}(\mathbf{x})]$ inside the square and
expanding:

$$= \underbrace{\left(f(\mathbf{x}) - \mathbb{E}[\hat{f}(\mathbf{x})]\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\left[\left(\hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(\mathbf{x})]\right)^2\right]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible noise}}$$

(the cross terms vanish because $\mathbb{E}[\epsilon] = 0$ and
$\mathbb{E}[\hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(\mathbf{x})]] = 0$ by
definition). This gives the compact decomposition:

$$\mathbb{E}\left[(y - \hat{f}(\mathbf{x}))^2\right] = \text{Bias}(\hat{f}(\mathbf{x}))^2 + \text{Var}(\hat{f}(\mathbf{x})) + \sigma^2$$

<a name="bv-interpretation"></a>
### Interpretation

- **Bias**: how far the model's average prediction (over many training sets) is
  from the true function $f$. High bias means the model is too simple
  (underfitting) — e.g. a linear model fit to curved data
- **Variance**: how much the model's prediction changes across different
  training sets. High variance means the model is too flexible (overfitting) —
  e.g. a deep, unpruned decision tree
- **Irreducible noise** $\sigma^2$: no model can reduce this; it is inherent to
  the data-generating process

Deep decision trees are the canonical high-variance, low-bias model — they can
fit training data almost perfectly (low bias) but change drastically with small
perturbations to the training set (high variance). This is exactly the profile
that ensembling is designed to fix.


In [ ]:
# Empirically demonstrate the bias-variance decomposition using repeated resampling
np.random.seed(42)

def true_function(x):
    return np.sin(1.5 * np.pi * x)

def generate_data(n_samples, noise_std=0.3):
    x = np.sort(np.random.uniform(0, 1, n_samples))
    y = true_function(x) + np.random.normal(0, noise_std, n_samples)
    return x.reshape(-1, 1), y

x_eval = np.linspace(0, 1, 100).reshape(-1, 1)
y_true = true_function(x_eval.flatten())

n_trials = 200
n_train = 30

def fit_and_predict(model_fn):
    predictions = np.zeros((n_trials, len(x_eval)))
    for trial in range(n_trials):
        X_train, y_train = generate_data(n_train)
        model = model_fn()
        model.fit(X_train, y_train)
        predictions[trial] = model.predict(x_eval)
    return predictions

# A high-variance model (deep tree) vs a high-bias model (shallow tree/stump)
deep_tree_preds = fit_and_predict(lambda: DecisionTreeRegressor(max_depth=None))
shallow_tree_preds = fit_and_predict(lambda: DecisionTreeRegressor(max_depth=1))

def bias_variance(predictions, y_true):
    mean_pred = predictions.mean(axis=0)
    bias_sq = np.mean((mean_pred - y_true) ** 2)
    variance = np.mean(predictions.var(axis=0))
    return bias_sq, variance

deep_bias_sq, deep_var = bias_variance(deep_tree_preds, y_true)
shallow_bias_sq, shallow_var = bias_variance(shallow_tree_preds, y_true)

print("\n" + "="*70)
print("EMPIRICAL BIAS-VARIANCE DECOMPOSITION")
print("="*70)
print(f"\n{'Model':<25}{'Bias^2':<15}{'Variance':<15}{'Bias^2 + Var':<15}")
print("-"*70)
print(f"{'Deep tree (depth=None)':<25}{deep_bias_sq:<15.4f}{deep_var:<15.4f}{deep_bias_sq+deep_var:<15.4f}")
print(f"{'Shallow tree (depth=1)':<25}{shallow_bias_sq:<15.4f}{shallow_var:<15.4f}{shallow_bias_sq+shallow_var:<15.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, preds, title in zip(axes, [deep_tree_preds, shallow_tree_preds],
                             ['Deep Tree: Low Bias, High Variance', 'Shallow Tree: High Bias, Low Variance']):
    for i in range(30):
        ax.plot(x_eval, preds[i], color='steelblue', alpha=0.1)
    ax.plot(x_eval, preds.mean(axis=0), color='red', linewidth=2, label='Mean prediction')
    ax.plot(x_eval, y_true, color='black', linewidth=2, linestyle='--', label='True function')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nThe deep tree's individual fits (thin blue lines) vary wildly across")
print("training samples -- high variance -- but their average tracks the true")
print("function well -- low bias. The shallow tree is stable across samples")
print("(low variance) but systematically misses the curve's shape (high bias).")


<a name="bagging"></a>
## Bagging

<a name="bootstrap-sampling"></a>
### Bootstrap Sampling

**Bootstrap aggregating (bagging)** trains $M$ models, each on an independent
**bootstrap sample** — a sample of size $N$ drawn with replacement from the
original $N$ training points. Each bootstrap sample contains roughly 63.2% of
the unique original points (some points appear multiple times, others not at
all):

$$P(\text{point included}) = 1 - \left(1 - \frac{1}{N}\right)^N \xrightarrow{N \to \infty} 1 - e^{-1} \approx 0.632$$

<a name="why-averaging-reduces-variance"></a>
### Why Averaging Reduces Variance

Suppose we average $M$ models $\hat{f}_1, \ldots, \hat{f}_M$, each with
variance $\sigma^2$ and pairwise correlation $\rho$ between any two models'
predictions. The variance of the average is:

$$\text{Var}\left(\frac{1}{M}\sum_{m=1}^{M} \hat{f}_m(\mathbf{x})\right) = \rho \sigma^2 + \frac{1-\rho}{M}\sigma^2$$

**Derivation.** For $M$ identically distributed variables with variance
$\sigma^2$ and pairwise correlation $\rho$:

$$\text{Var}\left(\frac{1}{M}\sum_m \hat{f}_m\right) = \frac{1}{M^2}\left(\sum_m \text{Var}(\hat{f}_m) + \sum_{m \neq m'} \text{Cov}(\hat{f}_m, \hat{f}_{m'})\right) = \frac{1}{M^2}\left(M\sigma^2 + M(M-1)\rho\sigma^2\right)$$

which simplifies to $\rho\sigma^2 + \frac{1-\rho}{M}\sigma^2$ as claimed. As
$M \to \infty$, the second term vanishes and the variance floor is $\rho\sigma^2$
— **averaging cannot reduce variance below the level set by inter-model
correlation**. This is why bagging alone (correlated trees, since they are all
grown on overlapping bootstrap samples of the same features) has a limited
variance-reduction ceiling, motivating Random Forest's extra decorrelation step.


In [ ]:
# Verify the bootstrap-inclusion probability and the variance-of-average formula empirically
np.random.seed(42)
N = 500
n_trials = 2000

inclusion_counts = np.zeros(n_trials)
for trial in range(n_trials):
    bootstrap_idx = np.random.randint(0, N, size=N)
    inclusion_counts[trial] = len(np.unique(bootstrap_idx)) / N

print("\n" + "="*70)
print("BOOTSTRAP SAMPLE COVERAGE")
print("="*70)
print(f"\nTheoretical: 1 - 1/e = {1 - np.exp(-1):.4f}")
print(f"Empirical (N={N}, {n_trials} trials): {inclusion_counts.mean():.4f} +/- {inclusion_counts.std():.4f}")

# Variance-of-average formula check with synthetic correlated predictors
rho_true = 0.3
sigma_sq = 1.0
M_values = [1, 5, 10, 50, 200]

cov_matrix_fn = lambda M: rho_true * sigma_sq * np.ones((M, M)) + (1 - rho_true) * sigma_sq * np.eye(M)

print(f"\n{'M':<10}{'Theoretical Var':<20}{'Formula: rho*sigma^2 + (1-rho)/M*sigma^2':<20}")
print("-"*70)
for M in M_values:
    cov = cov_matrix_fn(M)
    var_of_average = cov.sum() / (M ** 2)
    formula_value = rho_true * sigma_sq + (1 - rho_true) / M * sigma_sq
    print(f"{M:<10}{var_of_average:<20.4f}{formula_value:<20.4f}")


<a name="random-forest-decorrelating-the-ensemble"></a>
## Random Forest: Decorrelating the Ensemble

Random Forest extends bagging with one additional randomization: at each split
in each tree, only a random subset of $k < d$ features is considered (typically
$k = \sqrt{d}$ for classification). This deliberately weakens each individual
tree but **decorrelates** the trees in the ensemble — since different trees
cannot all rely on the same dominant feature, their errors become less
correlated. Referring back to the variance-of-average formula, lowering $\rho$
lowers the irreducible $\rho\sigma^2$ floor, so Random Forest achieves lower
ensemble variance than plain bagging even though each individual tree is
slightly weaker (slightly higher $\sigma^2$ and bias).

**Feature importance** in Random Forest is typically computed as the total
reduction in impurity (e.g. Gini impurity) attributable to a feature, averaged
across all trees and all splits using that feature — features used near the
root of many trees, on average, contribute more.


In [ ]:
# Demonstrate decorrelation: compare average pairwise correlation of tree
# predictions in bagging (all features) vs Random Forest (feature subsampling)
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

n_trees = 30

def get_tree_predictions(use_feature_subsampling):
    preds = np.zeros((n_trees, len(X_test)))
    n_features = X_train.shape[1]
    max_features = int(np.sqrt(n_features)) if use_feature_subsampling else n_features

    for t in range(n_trees):
        rng = np.random.RandomState(t)
        bootstrap_idx = rng.randint(0, len(X_train), len(X_train))
        X_boot, y_boot = X_train[bootstrap_idx], y_train[bootstrap_idx]
        tree = DecisionTreeClassifier(max_features=max_features, random_state=t)
        tree.fit(X_boot, y_boot)
        preds[t] = tree.predict_proba(X_test)[:, 1]
    return preds

bagging_preds = get_tree_predictions(use_feature_subsampling=False)
rf_preds = get_tree_predictions(use_feature_subsampling=True)

def mean_pairwise_correlation(preds):
    corr_matrix = np.corrcoef(preds)
    off_diag = corr_matrix[~np.eye(corr_matrix.shape[0], dtype=bool)]
    return off_diag.mean()

bagging_corr = mean_pairwise_correlation(bagging_preds)
rf_corr = mean_pairwise_correlation(rf_preds)

print("\n" + "="*70)
print("TREE CORRELATION: BAGGING vs RANDOM FOREST")
print("="*70)
print(f"\nMean pairwise correlation (bagging, all features):        {bagging_corr:.4f}")
print(f"Mean pairwise correlation (Random Forest, sqrt(d) features): {rf_corr:.4f}")
print("\nLower correlation means each tree's errors are more independent,")
print("so averaging (per the formula above) achieves a lower variance floor.")


In [ ]:
# Random Forest feature importance: total impurity reduction attributable
# to each feature, averaged across all trees and splits
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_train, y_train)

importances = rf_full.feature_importances_
feature_names = load_breast_cancer().feature_names
top_n = 10
top_idx = np.argsort(importances)[-top_n:]

fig, ax = plt.subplots(1, 1, figsize=(9, 6))
ax.barh(range(top_n), importances[top_idx], color='seagreen')
ax.set_yticks(range(top_n))
ax.set_yticklabels(feature_names[top_idx])
ax.set_xlabel('Feature importance (mean impurity decrease)')
ax.set_title('Random Forest Feature Importance (Top 10)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("="*70)
print(f"\nMost important feature: '{feature_names[top_idx[-1]]}' "
      f"(importance = {importances[top_idx[-1]]:.4f})")
print(f"Sum of all importances: {importances.sum():.4f} (normalized to 1 across all features)")
print("\nImportance is computed per tree as the total Gini-impurity decrease")
print("attributable to splits on that feature, averaged across all trees in")
print("the forest -- a feature used near the root of many trees, on average,")
print("contributes more than one used rarely or only in deep, low-impact splits.")


<a name="adaboost"></a>
## AdaBoost

<a name="the-boosting-idea"></a>
### The Boosting Idea

Where bagging trains models independently and in parallel, **boosting** trains
models sequentially: each new model focuses on the examples the current
ensemble gets wrong. AdaBoost (Adaptive Boosting) does this by maintaining a
distribution of weights over training examples, increasing the weight of
misclassified points before training the next weak learner.

<a name="weighted-weak-learners"></a>
### Weighted Weak Learners

A **weak learner** needs only to do slightly better than random guessing
(error rate $< 0.5$ for binary classification) — AdaBoost's decision stumps
(depth-1 trees) are a canonical weak learner. At iteration $m$, given sample
weights $w_{i,m}$, the weak learner $h_m$ is trained to minimize the
**weighted classification error**:

$$\text{err}_m = \frac{\sum_{i=1}^{N} w_{i,m} \, \mathbb{1}[y_i \neq h_m(\mathbf{x}_i)]}{\sum_{i=1}^{N} w_{i,m}}$$

<a name="deriving-the-weight-update-rule"></a>
### Deriving the Weight-Update Rule

AdaBoost assigns the weak learner a vote weight:

$$\alpha_m = \frac{1}{2} \ln\left(\frac{1 - \text{err}_m}{\text{err}_m}\right)$$

A learner with error close to $0$ gets a large positive vote weight; a learner
with error close to $0.5$ (no better than chance) gets a vote weight near $0$.
Sample weights are then updated:

$$w_{i,m+1} = w_{i,m} \exp\left(-\alpha_m y_i h_m(\mathbf{x}_i)\right)$$

with $y_i, h_m(\mathbf{x}_i) \in \{-1, +1\}$. When $h_m$ classifies $x_i$
correctly, $y_i h_m(\mathbf{x}_i) = 1$ and the weight shrinks by
$e^{-\alpha_m}$; when it misclassifies, $y_i h_m(\mathbf{x}_i) = -1$ and the
weight grows by $e^{\alpha_m}$. Weights are renormalized to sum to 1 after
each update.

<a name="connection-to-exponential-loss"></a>
### Connection to Exponential Loss

AdaBoost's weight update is not an ad-hoc heuristic — it is exactly the update
produced by greedily minimizing the **exponential loss**
$L(y, F(\mathbf{x})) = \exp(-y F(\mathbf{x}))$ for the ensemble
$F_m = \sum_{k=1}^{m} \alpha_k h_k$. At step $m$, we choose $(\alpha_m, h_m)$
to minimize:

$$\sum_{i=1}^{N} \exp\left(-y_i \left(F_{m-1}(\mathbf{x}_i) + \alpha_m h_m(\mathbf{x}_i)\right)\right) = \sum_{i=1}^{N} w_{i,m} \exp\left(-\alpha_m y_i h_m(\mathbf{x}_i)\right)$$

where $w_{i,m} = \exp(-y_i F_{m-1}(\mathbf{x}_i))$ is exactly the sample
weight carried forward from the previous iteration. Splitting the sum over
correctly and incorrectly classified points and differentiating with respect
to $\alpha_m$, setting the derivative to zero, recovers precisely
$\alpha_m = \frac{1}{2}\ln\left(\frac{1-\text{err}_m}{\text{err}_m}\right)$ —
AdaBoost is exact greedy (stagewise) minimization of exponential loss.

<a name="training-error-bound"></a>
### Training Error Bound

A classical result bounds the training error of the AdaBoost ensemble after
$M$ rounds:

$$\text{Training Error} \leq \prod_{m=1}^{M} 2\sqrt{\text{err}_m (1 - \text{err}_m)}$$

Since each factor $2\sqrt{\text{err}_m(1-\text{err}_m)} < 1$ whenever
$\text{err}_m < 0.5$ (better than chance), the bound shrinks **exponentially**
with the number of rounds — this is the formal justification for why boosting
even weak learners (barely better than a coin flip) can drive training error
to zero.


In [ ]:
print("\n" + "="*70)
print("ADABOOST ALGORITHM SUMMARY")
print("="*70)
print("\n1. Initialize weights: w_i = 1/N for all i")
print("2. For m = 1 to M:")
print("   a. Fit weak learner h_m to minimize weighted error")
print("   b. Compute err_m = weighted misclassification rate")
print("   c. Compute alpha_m = 0.5 * ln((1 - err_m) / err_m)")
print("   d. Update weights: w_i *= exp(-alpha_m * y_i * h_m(x_i)), then renormalize")
print("3. Final classifier: H(x) = sign(sum_m alpha_m * h_m(x))")
print("\nThis is exactly stagewise minimization of exponential loss --")
print("not a heuristic reweighting scheme.")


<a name="gradient-boosting"></a>
## Gradient Boosting

<a name="boosting-as-functional-gradient-descent"></a>
### Boosting as Functional Gradient Descent

Gradient boosting generalizes AdaBoost's idea to **any differentiable loss
function** $L(y, F(\mathbf{x}))$, not just exponential loss. The ensemble is
built additively:

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \gamma_m h_m(\mathbf{x})$$

At each step, instead of deriving a closed-form weight update (as AdaBoost
does for exponential loss), gradient boosting treats this as **gradient
descent in function space**: it computes the negative gradient of the loss
with respect to the current predictions,

$$r_{i,m} = -\left.\frac{\partial L(y_i, F(\mathbf{x}_i))}{\partial F(\mathbf{x}_i)}\right|_{F=F_{m-1}}$$

and fits the next weak learner $h_m$ to approximate these **pseudo-residuals**
$r_{i,m}$, then chooses the step size $\gamma_m$ by line search.

<a name="fitting-residuals"></a>
### Fitting Residuals

For squared-error loss $L(y, F) = \frac{1}{2}(y - F)^2$, the negative gradient
is simply the ordinary residual:

$$r_{i,m} = -\frac{\partial}{\partial F}\left[\frac{1}{2}(y_i - F(\mathbf{x}_i))^2\right]_{F=F_{m-1}} = y_i - F_{m-1}(\mathbf{x}_i)$$

so "fit a tree to the residuals" is the squared-error special case of the
general functional-gradient-descent framework. For other losses (log-loss for
classification, Huber loss for robust regression), the pseudo-residual takes a
different form, but the algorithm structure — fit a weak learner to the
negative gradient, add it to the ensemble with a step size — stays the same.
This is precisely what distinguishes gradient boosting (Friedman, 2001) from
AdaBoost: AdaBoost is the exponential-loss special case, discovered first;
gradient boosting is the general framework, discovered by recognizing the
connection to functional gradient descent.


In [ ]:
print("\n" + "="*70)
print("GRADIENT BOOSTING: GENERAL FRAMEWORK")
print("="*70)
print("\nF_0(x) = initial guess (e.g. mean of y)")
print("For m = 1 to M:")
print("  1. Compute pseudo-residuals: r_i = -dL(y_i, F)/dF at F = F_{m-1}(x_i)")
print("  2. Fit weak learner h_m(x) to approximate r_i")
print("  3. Choose step size gamma_m (line search)")
print("  4. Update: F_m(x) = F_{m-1}(x) + gamma_m * h_m(x)")
print("\nSquared-error loss: r_i = y_i - F_{m-1}(x_i)  (ordinary residual)")
print("AdaBoost's exponential loss: recovers the alpha_m/weight-update formulas")
print("derived above -- a special case of this general framework.")


<a name="from-scratch-implementation"></a>
## From-Scratch Implementation

<a name="decision-stumps"></a>
### Decision Stumps

In [ ]:
class DecisionStump:
    """
    A decision stump: a depth-1 decision tree. Splits on a single feature at a
    single threshold, weighted by per-sample weights. This is AdaBoost's
    canonical weak learner.
    """

    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.polarity = 1  # +1: predict +1 above threshold; -1: predict +1 below threshold
        self.alpha = None

    def fit(self, X, y, sample_weights):
        n_samples, n_features = X.shape
        best_error = np.inf

        for feature_idx in range(n_features):
            feature_values = np.unique(X[:, feature_idx])
            thresholds = (feature_values[:-1] + feature_values[1:]) / 2 if len(feature_values) > 1 else feature_values

            for threshold in thresholds:
                for polarity in [1, -1]:
                    predictions = np.ones(n_samples)
                    if polarity == 1:
                        predictions[X[:, feature_idx] < threshold] = -1
                    else:
                        predictions[X[:, feature_idx] >= threshold] = -1

                    weighted_error = np.sum(sample_weights[predictions != y])

                    if weighted_error < best_error:
                        best_error = weighted_error
                        self.feature_index = feature_idx
                        self.threshold = threshold
                        self.polarity = polarity

        return best_error

    def predict(self, X):
        n_samples = X.shape[0]
        predictions = np.ones(n_samples)
        if self.polarity == 1:
            predictions[X[:, self.feature_index] < self.threshold] = -1
        else:
            predictions[X[:, self.feature_index] >= self.threshold] = -1
        return predictions


print("\n" + "="*70)
print("DECISION STUMP: MINIMAL WEAK LEARNER")
print("="*70)
print("\nA stump splits on ONE feature at ONE threshold. Fitting tries every")
print("feature and every candidate threshold, keeping the split that minimizes")
print("the WEIGHTED misclassification error -- not the unweighted error, since")
print("AdaBoost needs the weak learner to focus on currently-hard examples.")


<a name="adaboost-in-numpy"></a>
### AdaBoost in NumPy

In [ ]:
class AdaBoostScratch:
    """
    AdaBoost using decision stumps as weak learners, implemented directly
    from the derivation above.
    """

    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.stumps = []
        self.alphas = []
        self.training_errors = []

    def fit(self, X, y):
        n_samples = X.shape[0]
        sample_weights = np.full(n_samples, 1 / n_samples)

        for m in range(self.n_estimators):
            stump = DecisionStump()
            weighted_error = stump.fit(X, y, sample_weights)

            # Guard against a perfect or degenerate weak learner (avoids log(0))
            weighted_error = np.clip(weighted_error, 1e-10, 1 - 1e-10)

            alpha = 0.5 * np.log((1 - weighted_error) / weighted_error)
            predictions = stump.predict(X)

            # Weight update derived above: w_i *= exp(-alpha * y_i * h_m(x_i))
            sample_weights *= np.exp(-alpha * y * predictions)
            sample_weights /= sample_weights.sum()  # renormalize

            self.stumps.append(stump)
            self.alphas.append(alpha)

            ensemble_pred = self.predict(X)
            self.training_errors.append(np.mean(ensemble_pred != y))

        return self

    def predict(self, X):
        stump_preds = np.array([stump.predict(X) for stump in self.stumps])
        weighted_sum = np.dot(self.alphas, stump_preds)
        return np.sign(weighted_sum)


print("\n" + "="*70)
print("ADABOOST: FROM-SCRATCH IMPLEMENTATION")
print("="*70)
print("\nEach round: fit a stump on weighted data, compute alpha, update")
print("and renormalize sample weights. Final prediction: sign of the")
print("alpha-weighted sum of all stumps' votes.")


<a name="visualization-and-interpretation"></a>
## Visualization and Interpretation

<a name="bias-variance-tradeoff-in-practice"></a>
### Bias-Variance Tradeoff in Practice

In [ ]:
# Train from-scratch AdaBoost on the breast cancer dataset
y_train_pm1 = np.where(y_train == 1, 1, -1)
y_test_pm1 = np.where(y_test == 1, 1, -1)

adaboost = AdaBoostScratch(n_estimators=50)
adaboost.fit(X_train, y_train_pm1)

train_acc = accuracy_score(y_train_pm1, adaboost.predict(X_train))
test_acc = accuracy_score(y_test_pm1, adaboost.predict(X_test))

print("\n" + "="*70)
print("FROM-SCRATCH ADABOOST RESULTS (Breast Cancer Dataset)")
print("="*70)
print(f"\nTraining accuracy: {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(range(1, len(adaboost.training_errors) + 1), adaboost.training_errors,
        linewidth=2, color='darkred')
ax.set_xlabel('Boosting round (m)')
ax.set_ylabel('Training error')
ax.set_title('AdaBoost Training Error vs Number of Rounds')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTraining error drops sharply in early rounds and then flattens near")
print("zero -- consistent with the exponential error bound derived above,")
print("where each round's factor 2*sqrt(err_m*(1-err_m)) shrinks the bound.")


<a name="adaboost-weight-evolution"></a>
### AdaBoost Weight Evolution

In [ ]:
# Visualize how AdaBoost re-weights hard examples on a simple 2D synthetic problem
np.random.seed(42)
X_2d = np.random.randn(150, 2)
y_2d = np.where(X_2d[:, 0] * X_2d[:, 1] > 0, 1, -1)  # XOR-like pattern: hard for a single stump
# Add label noise to a few points to create genuinely "hard" examples
noise_idx = np.random.choice(150, 15, replace=False)
y_2d[noise_idx] *= -1

adaboost_2d = AdaBoostScratch(n_estimators=3)

# Manually track weights through 3 rounds for visualization
sample_weights = np.full(150, 1 / 150)
weight_history = [sample_weights.copy()]
for m in range(3):
    stump = DecisionStump()
    weighted_error = stump.fit(X_2d, y_2d, sample_weights)
    weighted_error = np.clip(weighted_error, 1e-10, 1 - 1e-10)
    alpha = 0.5 * np.log((1 - weighted_error) / weighted_error)
    predictions = stump.predict(X_2d)
    sample_weights = sample_weights * np.exp(-alpha * y_2d * predictions)
    sample_weights /= sample_weights.sum()
    weight_history.append(sample_weights.copy())

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for round_idx, ax in enumerate(axes):
    weights = weight_history[round_idx]
    sizes = weights * 150 * 100  # scale for visibility
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap='coolwarm', s=sizes,
                          edgecolors='k', alpha=0.7)
    ax.set_title(f'Round {round_idx}: Sample Weights' if round_idx > 0 else 'Initial (uniform) weights')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nPoint size encodes sample weight. Misclassified/noisy points grow in")
print("weight round over round -- exactly the mechanism that makes the next")
print("weak learner focus on previously hard examples.")


<a name="comparing-ensemble-approaches"></a>
## Comparing Ensemble Approaches

In [ ]:
# Compare single tree, bagging, Random Forest, AdaBoost on the same dataset
models = {
    'Single Decision Tree': DecisionTreeClassifier(random_state=42),
    'Bagging (50 trees)': BaggingClassifier(n_estimators=50, random_state=42),
    'Random Forest (50 trees)': RandomForestClassifier(n_estimators=50, random_state=42),
    'AdaBoost (sklearn, 50 stumps)': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=42
    ),
    'AdaBoost (from-scratch, 50 stumps)': None,  # handled separately
}

results = []
for name, model in models.items():
    if name == 'AdaBoost (from-scratch, 50 stumps)':
        train_a = accuracy_score(y_train_pm1, adaboost.predict(X_train))
        test_a = accuracy_score(y_test_pm1, adaboost.predict(X_test))
    else:
        model.fit(X_train, y_train)
        train_a = accuracy_score(y_train, model.predict(X_train))
        test_a = accuracy_score(y_test, model.predict(X_test))
    results.append((name, train_a, test_a))

print("\n" + "="*70)
print("ENSEMBLE METHOD COMPARISON (Breast Cancer Dataset)")
print("="*70)
print(f"\n{'Model':<35}{'Train Accuracy':<18}{'Test Accuracy':<18}")
print("-"*70)
for name, train_a, test_a in results:
    print(f"{name:<35}{train_a:<18.4f}{test_a:<18.4f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
names = [r[0] for r in results]
test_accs = [r[2] for r in results]
ax.barh(names, test_accs, color='steelblue')
ax.set_xlabel('Test Accuracy')
ax.set_title('Ensemble Method Comparison')
ax.set_xlim(0.85, 1.0)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nThe from-scratch AdaBoost implementation achieves accuracy comparable")
print("to scikit-learn's AdaBoostClassifier, confirming the derivation and")
print("implementation are correct. All ensemble methods outperform the single")
print("decision tree, confirming ensembling reduces variance as derived above.")


<a name="conclusion"></a>
## Conclusion

<a name="key-insights-3"></a>
### Key Insights

1. **The bias-variance decomposition** splits expected error into
   bias-squared, variance, and irreducible noise — ensembling primarily
   targets the variance term

2. **Bagging** averages independently trained models on bootstrap samples;
   the variance-of-average formula shows the benefit is capped by
   inter-model correlation $\rho$

3. **Random Forest** decorrelates trees via random feature subsampling at
   each split, pushing the variance floor lower than plain bagging achieves

4. **AdaBoost** is not a heuristic reweighting scheme — it is exact stagewise
   minimization of exponential loss, with a training error bound that
   shrinks exponentially with the number of rounds

5. **Gradient boosting** generalizes AdaBoost's idea to arbitrary
   differentiable losses via functional gradient descent; "fit the
   residuals" is the squared-error special case

6. **The from-scratch AdaBoost implementation** matches scikit-learn's
   performance, confirming the weight-update derivation is correctly implemented


<a name="when-to-use-each-approach"></a>
### When to Use Each Approach

**Bagging / Random Forest are particularly good for:**
- High-variance base learners (deep trees) where the goal is variance reduction
- Problems where parallel training is valuable (bagging trains independently)
- Robustness to noisy training data (averaging dilutes individual outlier influence)

**Boosting (AdaBoost / gradient boosting) is particularly good for:**
- High-bias base learners (shallow trees, stumps) where the goal is bias reduction
- Problems with a clear notion of "hard" examples worth focusing on
- State-of-the-art tabular data performance (gradient boosting variants dominate
  Kaggle-style competitions) — covered in Lesson 7b with XGBoost and LightGBM

**Caution**: boosting can overfit to label noise, since noisy/mislabeled points
keep receiving high weight round after round — bagging is generally more
robust when training labels are unreliable.


<a name="further-reading-3"></a>
### Further Reading

**Foundational Papers:**
- Freund, Y., & Schapire, R. E. (1997). "A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting"
- Friedman, J. H. (2001). "Greedy Function Approximation: A Gradient Boosting Machine"
- Breiman, L. (2001). "Random Forests"

**Comprehensive References:**
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). "The Elements of Statistical Learning (ESL)", Chapter 10 (Boosting and Additive Trees), Chapter 15 (Random Forests)
- Schapire, R. E., & Freund, Y. (2012). "Boosting: Foundations and Algorithms"
